# Graph RAG Pipeline

A Graph RAG (Retrieval-Augmented Generation) system that combines knowledge graph traversal with vector similarity search to answer questions with structured relational context.


## 1. Install Dependencies

Installs all required packages: LangChain ecosystem, Google Generative AI, Qdrant vector store, and Wikipedia loader.


In [36]:
!pip install wikipedia langchain-text-splitters langchain-google-genai langchain_community langchain-qdrant qdrant-client -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 10.2 MB/s eta 0:00:00


## 2. Import Dependencies

Imports the libraries needed

In [7]:
from pydantic import BaseModel
import httpx
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from typing import List
import networkx as nx

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

## 3. Fetch News Articles (Data Source)

Fetches the latest news articles on **Artificial Intelligence** using NewsAPI. Each article's title, description, and content are combined into a single text block with source metadata. Limited to 10 articles for the prototype.

In [16]:
API_KEY = "API_KEY"
TOPIC = "artificial intelligence"

url = f"https://newsapi.org/v2/everything?q={TOPIC}&sortBy=publishedAt&apiKey={API_KEY}"

# Best practice with httpx: use a client block to handle connection pooling efficiently
with httpx.Client() as client:
    response = client.get(url)

if response.status_code == 200:
    data = response.json()
    articles = data.get("articles", [])

    rag_documents = []
    for art in articles[:10]:
        text_to_chunk = f"Title: {art['title']}\nDescription: {art['description']}\nContent: {art['content']}"
        metadata = {"source": art['url'], "published": art['publishedAt']}
        rag_documents.append({"text": text_to_chunk, "metadata": metadata})

    print(f"Successfully formatted {len(rag_documents)} articles for ingestion.")
else:
    print(f"Failed to fetch data: {response.status_code}")

Successfully formatted 10 articles for ingestion.


In [22]:
docs = []
for doc in rag_documents:
  docs.append(Document(page_content=doc['text'], metadata=doc['metadata']))

## 4. Chunk Documents

Splits documents into overlapping chunks of 768 characters with 30-character overlap. Overlap ensures context is not lost at chunk boundaries.

In [23]:
splitter = RecursiveCharacterTextSplitter(chunk_size=768, chunk_overlap=30)
texts = splitter.split_documents(docs)

## 5. Define Graph Schema

Defines Pydantic schemas for structured LLM output. `Relation` represents a directed edge `(src, rel, dst)`. `GraphData` holds a list of entities and relations extracted from a chunk.

In [24]:
class Relation(BaseModel):
    src: str
    rel: str
    dst: str

class GraphData(BaseModel):
    entities: List[str]
    relations: List[Relation]

## 6. Initialize LLM

Initializes Gemini Flash Lite as the base LLM and wraps it with structured output to enforce `GraphData` schema — guarantees valid entity/relation extraction with no JSON parsing needed.

In [31]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite",api_key="GEMINI_API_KEY")
structured_llm = llm.with_structured_output(GraphData)

## 7. Define Extraction Prompt

Prompt template instructing the LLM to extract named entities (nouns: people, diseases, drugs, organs, concepts) and their relationships from a given text chunk.

In [28]:
EXTRACT_PROMPT = """
Extract all entities and relationships from the text below.
Entities are nouns: people, diseases, drugs, organs, concepts.
Relations describe how two entities are connected.

Text: {text}
"""

## 8. Entity & Relation Extraction Function

Invokes the structured LLM on a text chunk and returns a `GraphData` object. Falls back to empty entities and relations if extraction fails.

In [32]:
def extract_graph(text: str) -> GraphData:
    try:
        return structured_llm.invoke(EXTRACT_PROMPT.format(text=text))
    except Exception as e:
        print(f"Extraction failed: {e}")
        return GraphData(entities=[], relations=[])

## 9. Build Knowledge Graph

Iterates over document chunks, extracts entities and relations, and builds a directed `NetworkX` graph. Each entity becomes a node; each relation becomes a labeled directed edge.

In [33]:
G = nx.DiGraph()

for i, chunk in enumerate(texts[:30]):  # limit to 30 chunks for prototype
    print(f"Processing chunk {i+1}/30...", end="\r")
    data = extract_graph(chunk.page_content)

    for entity in data.entities:
        G.add_node(entity)

    for rel in data.relations:
        G.add_edge(rel.src, rel.dst, relation=rel.rel)

print(f"\nDone. Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")


Done. Nodes: 78, Edges: 54


## 10. Initialize Vector Store (Qdrant)

Initializes Google's `gemini-embedding-2` model (3072-dim vectors) and sets up an in-memory Qdrant collection with cosine similarity for semantic chunk retrieval.

In [40]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    google_api_key="GEMINI_API_KEY"
)

client = QdrantClient(":memory:",force_recreate=True)

client.create_collection(
    collection_name="graph_rag",
    vectors_config=VectorParams(size=3072, distance=Distance.COSINE),

)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="graph_rag",
    embedding=embeddings
)

## 11. Embed & Index Documents

Embeds all document chunks and indexes them into Qdrant. This enables semantic similarity search at query time.

In [41]:
vector_store.add_documents(texts)
print(f"Qdrant ready. Total chunks: {len(texts)}")

Qdrant ready. Total chunks: 10


## 12. Query Entity Extraction

Extracts named entities from the user's question using a structured LLM call. These entities serve as entry points for graph traversal — different from `extract_graph` which operates on document chunks during ingestion.

In [47]:
class EntityList(BaseModel):
    entities: List[str]

entity_llm = llm.with_structured_output(EntityList)

def extract_query_entities(question: str) -> List[str]:
    prompt = f"""
    Extract entity names from this question.
    Entities are nouns: people, diseases, drugs, organs, concepts, technologies.
    Question: {question}
    """
    try:
        result = entity_llm.invoke(prompt)
        return result.entities
    except:
        return []

## 13. Graph Traversal

Takes query entities, fuzzy-matches them against graph nodes, and traverses up to 2 hops to retrieve a relevant subgraph. Returns the subgraph as human-readable triples: `Entity A --[relation]--> Entity B`.

In [48]:
def get_subgraph_context(entities: List[str], hops: int = 2) -> str:
    visited = set()
    for entity in entities:
        matched = [n for n in G.nodes if entity.lower() in n.lower()]
        for node in matched:
            visited.add(node)
            neighbors = nx.single_source_shortest_path(G, node, cutoff=hops)
            visited.update(neighbors.keys())

    subgraph = G.subgraph(visited)
    triples = []
    for u, v, data in subgraph.edges(data=True):
        triples.append(f"{u} --[{data.get('relation', 'related_to')}]--> {v}")

    return "\n".join(triples) if triples else "No graph context found."

## 14. Vector Search

Performs semantic similarity search over Qdrant to retrieve the top-K most relevant document chunks for the question.

In [49]:
def get_vector_context(question: str, k: int = 3) -> str:
    results = vector_store.similarity_search(question, k=k)
    return "\n\n".join([r.page_content for r in results])

## 15. Fusion & Answer Generation

The core query pipeline. Combines graph traversal context (structured relationships) with vector search context (semantic chunks) into a single prompt and generates a grounded answer via the LLM.

In [50]:
ANSWER_PROMPT = """
Use the context below to answer the question accurately.

=== Knowledge Graph (structured relationships) ===
{graph_context}

=== Retrieved Chunks (semantic search) ===
{vector_context}

Question: {question}
Answer:
"""

def graph_rag_query(question: str) -> str:
    entities = extract_query_entities(question)
    print(f"Entities: {entities}")

    graph_context = get_subgraph_context(entities)
    print(f"\nGraph context:\n{graph_context}\n")

    vector_context = get_vector_context(question)

    prompt = ANSWER_PROMPT.format(
        graph_context=graph_context,
        vector_context=vector_context,
        question=question
    )
    return llm.invoke(prompt).content

## 16. Run a Query

End-to-end test of the Graph RAG pipeline. Prints extracted entities, graph triples retrieved, and the final generated answer.

In [53]:
question = "What is the relationship between AI and machine learning?"
answer = graph_rag_query(question)
print(answer)

Entities: ['AI', 'machine learning']

Graph context:
AI-designed nuclear test flight vehicle --[debuted at]--> Great American State Fair
AI stock boom --[benefits]--> economy
AI stock boom --[increased demand for]--> electricity
AI stock boom --[impacted]--> construction companies
AI stock boom --[impacted]--> industrial stocks
productivity gains --[impacts]--> inflation
Proprietary AI --[is considered]--> expensive
Proprietary AI --[is considered]--> centralized
Aires Tide project --[used]--> AI
Aires Tide project --[used]--> supercomputing
Aires Tide project --[used]--> 3D printing
AI --[advances]--> nuclear tech
AI --[potential in]--> Medicine
AI --[potential in]--> Science

[{'type': 'text', 'text': 'The provided text does not contain information regarding the relationship between AI and machine learning. Therefore, the question cannot be answered using the provided context.', 'extras': {'signature': 'EjQKMgEMOdbHtXi9j4S7+YrSZ7Ih2DRSErGh8KB2JaX2/Y025J0LIN5fAuHqBGFUFKTkRwmW'}}]
